<a href="https://colab.research.google.com/github/najnin26/Quantum-Enhanced-Text-Classification/blob/main/Spam_Bilstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================
# 1. Imports
# ==============================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, RandomSampler, SequentialSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
import re
import nltk
from nltk.corpus import stopwords
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
nltk.download('stopwords', quiet=True)

torch.backends.cudnn.enabled = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ==============================
# 2. Config
# ==============================
MAX_LEN    = 128
BATCH_SIZE = 32
EPOCHS     = 10
LR         = 1e-3
HIDDEN_DIM = 128
VOCAB_SIZE = 128100
EMBED_DIM  = 64
MODEL_NAME = "microsoft/deberta-v3-base"

# ==============================
# 3. Load & Preprocess Data
# ==============================
df = pd.read_csv("/content/spamdata_v2 (1).csv")
df = df.dropna(subset=['text', 'label'])

stop_words = set(stopwords.words('english'))

def clean(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = text.lower()
    words = [w for w in text.split() if w not in stop_words]
    return " ".join(words)

df['text'] = df['text'].apply(clean)

le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])

# Balance classes
majority = df['label'].value_counts().idxmax()
minority = df['label'].value_counts().idxmin()
df_maj = df[df['label'] == majority].sample(len(df[df['label'] == minority]), random_state=42)
df = pd.concat([df_maj, df[df['label'] == minority]]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Balanced dataset: {df['label'].value_counts().to_dict()}")

# ==============================
# 4. Split
# ==============================
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'], df['label'], test_size=0.2, stratify=df['label'], random_state=42
)

# ==============================
# 5. Tokenization
# ==============================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(texts):
    return tokenizer(
        texts.tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

train_enc = encode(train_texts)
val_enc   = encode(val_texts)

def make_loader(enc, labels, sampler_cls):
    dataset = TensorDataset(
        enc['input_ids'],
        enc['attention_mask'],
        torch.tensor(labels.values, dtype=torch.long)
    )
    return DataLoader(dataset, sampler=sampler_cls(dataset), batch_size=BATCH_SIZE)

train_loader = make_loader(train_enc, train_labels, RandomSampler)
val_loader   = make_loader(val_enc,   val_labels,   SequentialSampler)

# ==============================
# 6. Metrics Helper
# ==============================
def compute_all_metrics(all_labels, all_preds, all_probs, name):
    # Guard against any remaining NaN
    all_probs = np.nan_to_num(np.array(all_probs), nan=0.5)

    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    rec  = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    spec = recall_score(all_labels, all_preds, pos_label=0, average='binary', zero_division=0)
    f1   = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    auc  = roc_auc_score(all_labels, all_probs)

    print(f"\n{'='*55}")
    print(f"  {name} — Final Results")
    print(f"{'='*55}")
    print(f"  Accuracy    : {acc:.4f}")
    print(f"  Precision   : {prec:.4f}")
    print(f"  Recall      : {rec:.4f}")
    print(f"  Specificity : {spec:.4f}")
    print(f"  F1-score    : {f1:.4f}")
    print(f"  AUC         : {auc:.4f}")
    print(f"\n{classification_report(all_labels, all_preds)}")
    return dict(accuracy=acc, precision=prec, recall=rec,
                specificity=spec, f1=f1, auc=auc)

# ==============================
# 7. Shared Training Loop
# ==============================
def train_and_evaluate(model, train_loader, val_loader, model_name, epochs=EPOCHS, lr=LR):
    model.to(DEVICE)

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer   = AdamW(trainable, lr=lr)
    total_steps = len(train_loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0, num_training_steps=total_steps
    )
    loss_fn = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(epochs):
        # ── Train ──
        model.train()
        t_loss, t_correct, t_total = 0, 0, 0
        for batch in train_loader:
            ids, mask, lbls = [b.to(DEVICE) for b in batch]
            optimizer.zero_grad()
            logits = model(ids, mask)
            loss   = loss_fn(logits, lbls)

            if torch.isnan(loss):
                print(f"   NaN loss detected — skipping batch")
                continue

            loss.backward()
            nn.utils.clip_grad_norm_(trainable, 1.0)
            optimizer.step()
            scheduler.step()
            t_loss    += loss.item()
            t_correct += (logits.argmax(1) == lbls).sum().item()
            t_total   += lbls.size(0)

        # ── Validate ──
        model.eval()
        v_loss, v_correct, v_total = 0, 0, 0
        all_preds, all_labels, all_probs = [], [], []

        with torch.no_grad():
            for batch in val_loader:
                ids, mask, lbls = [b.to(DEVICE) for b in batch]
                logits = model(ids, mask)
                loss   = loss_fn(logits, lbls)
                probs  = F.softmax(logits, dim=1)[:, 1].cpu().float().numpy()
                preds  = logits.argmax(1).cpu().numpy()

                v_loss    += loss.item() if not torch.isnan(loss) else 0
                v_correct += (logits.argmax(1) == lbls).sum().item()
                v_total   += lbls.size(0)
                all_preds.extend(preds)
                all_labels.extend(lbls.cpu().numpy())
                all_probs.extend(probs)

        avg_t = t_loss / max(len(train_loader), 1)
        avg_v = v_loss / max(len(val_loader), 1)
        v_acc = v_correct / max(v_total, 1)
        history['train_loss'].append(avg_t)
        history['val_loss'].append(avg_v)
        history['val_acc'].append(v_acc)

        print(f"[{model_name}] Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {avg_t:.4f} | Val Loss: {avg_v:.4f} | Val Acc: {v_acc:.4f}")

    metrics = compute_all_metrics(
        np.array(all_labels), np.array(all_preds), np.array(all_probs), model_name
    )
    return history, metrics

class BiLSTMOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=EMBED_DIM,
            hidden_size=HIDDEN_DIM // 4,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.7
        )
        self.dropout = nn.Dropout(0.7)
        self.fc      = nn.Linear(HIDDEN_DIM // 2, 2)

    def forward(self, ids, mask):
        x = self.embedding(ids).float()
        x = x * mask.unsqueeze(-1).float()
        lstm_out, _ = self.lstm(x)
        mask_exp = mask.unsqueeze(-1).float()
        pooled   = (lstm_out * mask_exp).sum(1) / mask_exp.sum(1).clamp(min=1e-9)
        return self.fc(self.dropout(pooled))

# ==============================
# 10. Run MODEL A — BiLSTM
# ==============================
print("\n" + "="*55)
print("MODEL A: BiLSTM (No Pretrained Encoder) ")
print("="*55)
bilstm_model = BiLSTMOnly()
history_a, metrics_a = train_and_evaluate(
    bilstm_model, train_loader, val_loader, "BiLSTM", lr=1e-5
)

# ==============================
# 12. Comparison Table
# ==============================
print("\n" + "="*55)
print("FINAL COMPARISON TABLE")
print("="*55)

keys       = ['accuracy', 'precision', 'recall', 'specificity', 'f1', 'auc']
labels_row = ['Accuracy', 'Precision', 'Recall', 'Specificity', 'F1-score', 'AUC']

comparison = pd.DataFrame({
    'Metric'          : labels_row,
    'BiLSTM'          : [round(metrics_a[k], 4) for k in keys],
    'QDeBERTa-BiLSTM' : [0.9666, 0.9664, 0.9668, 0.9668, 0.9665, 0.9886],
}).set_index('Metric')

print(comparison.to_string())

Using device: cuda
Balanced dataset: {1: 747, 0: 747}



MODEL A: BiLSTM (No Pretrained Encoder) 
[BiLSTM] Epoch 1/10 | Train Loss: 0.6980 | Val Loss: 0.6975 | Val Acc: 0.5017
[BiLSTM] Epoch 2/10 | Train Loss: 0.6994 | Val Loss: 0.6972 | Val Acc: 0.5017
[BiLSTM] Epoch 3/10 | Train Loss: 0.6990 | Val Loss: 0.6969 | Val Acc: 0.5017
[BiLSTM] Epoch 4/10 | Train Loss: 0.6979 | Val Loss: 0.6967 | Val Acc: 0.5017
[BiLSTM] Epoch 5/10 | Train Loss: 0.6991 | Val Loss: 0.6965 | Val Acc: 0.5017
[BiLSTM] Epoch 6/10 | Train Loss: 0.7005 | Val Loss: 0.6964 | Val Acc: 0.5017
[BiLSTM] Epoch 7/10 | Train Loss: 0.6973 | Val Loss: 0.6962 | Val Acc: 0.5017
[BiLSTM] Epoch 8/10 | Train Loss: 0.6981 | Val Loss: 0.6961 | Val Acc: 0.5017
[BiLSTM] Epoch 9/10 | Train Loss: 0.6978 | Val Loss: 0.6961 | Val Acc: 0.5017
[BiLSTM] Epoch 10/10 | Train Loss: 0.6958 | Val Loss: 0.6961 | Val Acc: 0.5017

  BiLSTM — Final Results
  Accuracy    : 0.5017
  Precision   : 0.2517
  Recall      : 0.5017
  Specificity : 1.0000
  F1-score    : 0.3352
  AUC         : 0.4649

            

In [ ]:
# ==============================
# 1. Imports
# ==============================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, RandomSampler, SequentialSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
import re
import nltk
from nltk.corpus import stopwords
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
nltk.download('stopwords', quiet=True)

torch.backends.cudnn.enabled = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ==============================
# 2. Config
# ==============================
MAX_LEN    = 128
BATCH_SIZE = 32
EPOCHS     = 10
LR         = 1e-3
HIDDEN_DIM = 128
VOCAB_SIZE = 128100
EMBED_DIM  = 64
MODEL_NAME = "microsoft/deberta-v3-base"

# ==============================
# 3. Load & Preprocess Data
# ==============================
df = pd.read_csv("/content/spamdata_v2 (1).csv")
df = df.dropna(subset=['text', 'label'])

stop_words = set(stopwords.words('english'))

def clean(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = text.lower()
    words = [w for w in text.split() if w not in stop_words]
    return " ".join(words)

df['text'] = df['text'].apply(clean)

le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])

# Balance classes
majority = df['label'].value_counts().idxmax()
minority = df['label'].value_counts().idxmin()
df_maj = df[df['label'] == majority].sample(len(df[df['label'] == minority]), random_state=42)
df = pd.concat([df_maj, df[df['label'] == minority]]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Balanced dataset: {df['label'].value_counts().to_dict()}")

# ==============================
# 4. Split
# ==============================
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'], df['label'], test_size=0.2, stratify=df['label'], random_state=42
)

# ==============================
# 5. Tokenization
# ==============================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(texts):
    return tokenizer(
        texts.tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

train_enc = encode(train_texts)
val_enc   = encode(val_texts)

def make_loader(enc, labels, sampler_cls):
    dataset = TensorDataset(
        enc['input_ids'],
        enc['attention_mask'],
        torch.tensor(labels.values, dtype=torch.long)
    )
    return DataLoader(dataset, sampler=sampler_cls(dataset), batch_size=BATCH_SIZE)

train_loader = make_loader(train_enc, train_labels, RandomSampler)
val_loader   = make_loader(val_enc,   val_labels,   SequentialSampler)


def compute_all_metrics(all_labels, all_preds, all_probs, name):
    all_probs = np.nan_to_num(np.array(all_probs), nan=0.5)

    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    rec  = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    spec = recall_score(all_labels, all_preds, pos_label=0, average='binary', zero_division=0)
    f1   = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    auc  = roc_auc_score(all_labels, all_probs)

    print(f"\n{'='*55}")
    print(f"  {name} — Final Results")
    print(f"{'='*55}")
    print(f"  Accuracy    : {acc:.4f}")
    print(f"  Precision   : {prec:.4f}")
    print(f"  Recall      : {rec:.4f}")
    print(f"  Specificity : {spec:.4f}")
    print(f"  F1-score    : {f1:.4f}")
    print(f"  AUC         : {auc:.4f}")
    print(f"\n{classification_report(all_labels, all_preds)}")
    return dict(accuracy=acc, precision=prec, recall=rec,
                specificity=spec, f1=f1, auc=auc)

def train_and_evaluate(model, train_loader, val_loader, model_name, epochs=EPOCHS, lr=LR):
    model.to(DEVICE)

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer   = AdamW(trainable, lr=lr)
    total_steps = len(train_loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0, num_training_steps=total_steps
    )
    loss_fn = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(epochs):
        # ── Train ──
        model.train()
        t_loss, t_correct, t_total = 0, 0, 0
        for batch in train_loader:
            ids, mask, lbls = [b.to(DEVICE) for b in batch]
            optimizer.zero_grad()
            logits = model(ids, mask)
            loss   = loss_fn(logits, lbls)

            if torch.isnan(loss):
                print(f"  NaN loss detected — skipping batch")
                continue

            loss.backward()
            nn.utils.clip_grad_norm_(trainable, 1.0)
            optimizer.step()
            scheduler.step()
            t_loss    += loss.item()
            t_correct += (logits.argmax(1) == lbls).sum().item()
            t_total   += lbls.size(0)

        # ── Validate ──
        model.eval()
        v_loss, v_correct, v_total = 0, 0, 0
        all_preds, all_labels, all_probs = [], [], []

        with torch.no_grad():
            for batch in val_loader:
                ids, mask, lbls = [b.to(DEVICE) for b in batch]
                logits = model(ids, mask)
                loss   = loss_fn(logits, lbls)
                probs  = F.softmax(logits, dim=1)[:, 1].cpu().float().numpy()
                preds  = logits.argmax(1).cpu().numpy()

                v_loss    += loss.item() if not torch.isnan(loss) else 0
                v_correct += (logits.argmax(1) == lbls).sum().item()
                v_total   += lbls.size(0)
                all_preds.extend(preds)
                all_labels.extend(lbls.cpu().numpy())
                all_probs.extend(probs)

        avg_t = t_loss / max(len(train_loader), 1)
        avg_v = v_loss / max(len(val_loader), 1)
        v_acc = v_correct / max(v_total, 1)
        history['train_loss'].append(avg_t)
        history['val_loss'].append(avg_v)
        history['val_acc'].append(v_acc)

        print(f"[{model_name}] Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {avg_t:.4f} | Val Loss: {avg_v:.4f} | Val Acc: {v_acc:.4f}")

    metrics = compute_all_metrics(
        np.array(all_labels), np.array(all_preds), np.array(all_probs), model_name
    )
    return history, metrics

class BiLSTMOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=EMBED_DIM,
            hidden_size=HIDDEN_DIM // 4,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.7
        )
        self.dropout = nn.Dropout(0.6)
        self.fc      = nn.Linear(HIDDEN_DIM // 2, 2)  # 32×2 → 64 → 2

    def forward(self, ids, mask):
        x = self.embedding(ids).float()
        x = x * mask.unsqueeze(-1).float()
        lstm_out, _ = self.lstm(x)
        mask_exp = mask.unsqueeze(-1).float()
        pooled   = (lstm_out * mask_exp).sum(1) / mask_exp.sum(1).clamp(min=1e-9)
        return self.fc(self.dropout(pooled))

# ==============================
# 10. Run MODEL A — BiLSTM
# ==============================
print("\n" + "="*55)
print("MODEL A: BiLSTM (No Pretrained Encoder) ")
print("="*55)
bilstm_model = BiLSTMOnly()
history_a, metrics_a = train_and_evaluate(
    bilstm_model, train_loader, val_loader, "BiLSTM", lr=1e-4
)

# ==============================
# 12. Comparison Table
# ==============================
print("\n" + "="*55)
print("FINAL COMPARISON TABLE")
print("="*55)

keys       = ['accuracy', 'precision', 'recall', 'specificity', 'f1', 'auc']
labels_row = ['Accuracy', 'Precision', 'Recall', 'Specificity', 'F1-score', 'AUC']

comparison = pd.DataFrame({
    'Metric'          : labels_row,
    'BiLSTM'          : [round(metrics_a[k], 4) for k in keys],
    'QDeBERTa-BiLSTM' : [0.9666, 0.9664, 0.9668, 0.9668, 0.9665, 0.9886],
}).set_index('Metric')

print(comparison.to_string())

Using device: cuda
Balanced dataset: {1: 747, 0: 747}



MODEL A: BiLSTM (No Pretrained Encoder) 
[BiLSTM] Epoch 1/10 | Train Loss: 0.6944 | Val Loss: 0.6906 | Val Acc: 0.5017
[BiLSTM] Epoch 2/10 | Train Loss: 0.6912 | Val Loss: 0.6882 | Val Acc: 0.5050
[BiLSTM] Epoch 3/10 | Train Loss: 0.6869 | Val Loss: 0.6861 | Val Acc: 0.5418
[BiLSTM] Epoch 4/10 | Train Loss: 0.6844 | Val Loss: 0.6840 | Val Acc: 0.6589
[BiLSTM] Epoch 5/10 | Train Loss: 0.6829 | Val Loss: 0.6821 | Val Acc: 0.7458
[BiLSTM] Epoch 6/10 | Train Loss: 0.6808 | Val Loss: 0.6804 | Val Acc: 0.7860
[BiLSTM] Epoch 7/10 | Train Loss: 0.6796 | Val Loss: 0.6788 | Val Acc: 0.7960
[BiLSTM] Epoch 8/10 | Train Loss: 0.6800 | Val Loss: 0.6776 | Val Acc: 0.7759
[BiLSTM] Epoch 9/10 | Train Loss: 0.6799 | Val Loss: 0.6769 | Val Acc: 0.7826
[BiLSTM] Epoch 10/10 | Train Loss: 0.6772 | Val Loss: 0.6766 | Val Acc: 0.7826

  BiLSTM — Final Results
  Accuracy    : 0.7826
  Precision   : 0.7972
  Recall      : 0.7826
  Specificity : 0.6733
  F1-score    : 0.7800
  AUC         : 0.8800

            